In [1]:
# 02_descriptive_analysis.ipynb
# Weighted descriptive analysis of catastrophic health expenditure (CHE):
# incidence trends, income-quintile gradient, the health-shock effect, and the
# raw private-insurance (PHI) buffer. All figures are grayscale, no titles/captions,
# saved as 600-dpi PNG and PDF. Tables are written to ../results/tables/.

import warnings, os
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt

PROC, FIG, TAB = "../data/processed", "../results/figures", "../results/tables"
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.3", "savefig.dpi": 600,
                     "figure.dpi": 110, "axes.grid": True, "grid.color": "0.9"})
GRAYS = ["0.15", "0.45", "0.70"]   # grayscale shades for up to three series

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"{FIG}/{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

def wmean(df, col, w="I_WGC"):
    d = df[[col, w]].dropna()
    return np.average(d[col], weights=d[w]) if len(d) else np.nan

p = pd.read_parquet(f"{PROC}/khp_panel.parquet")
years = sorted(p["year"].unique())

# ---- Figure 1: CHE incidence by year (10/25/40% thresholds) ----
rows = []
for y in years:
    g = p[p.year == y]
    rows.append([y] + [100 * wmean(g, f"che{t}") for t in (10, 25, 40)])
che_year = pd.DataFrame(rows, columns=["year", "CHE10", "CHE25", "CHE40"])
che_year.to_csv(f"{TAB}/t02_che_by_year.csv", index=False)
fig, ax = plt.subplots(figsize=(6, 4))
for (c, m, ls, lab) in [("CHE10", "o", "-", "10% threshold"), ("CHE25", "s", "--", "25% threshold"), ("CHE40", "^", ":", "40% threshold")]:
    ax.plot(che_year["year"], che_year[c], marker=m, linestyle=ls, color=GRAYS[["CHE10", "CHE25", "CHE40"].index(c)],
            lw=2, ms=7, label=lab, markerfacecolor="white", markeredgewidth=1.5)
ax.set_xlabel("Year"); ax.set_ylabel("Households with CHE (%)"); ax.legend(frameon=False)
save_fig(fig, "fig01_che_trend")

# ---- Figure 2: CHE 40% by income quintile ----
q_rows = [[q, 100 * wmean(p[p.inc_q == q], "che40")] for q in [1, 2, 3, 4, 5]]
che_q = pd.DataFrame(q_rows, columns=["income_quintile", "CHE40"])
che_q.to_csv(f"{TAB}/t02_che_by_quintile.csv", index=False)
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar([f"Q{q}" for q in che_q["income_quintile"]], che_q["CHE40"],
              color=["0.2" if q == 1 else "0.6" for q in che_q["income_quintile"]], edgecolor="0.3")
for b, v in zip(bars, che_q["CHE40"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.15, f"{v:.1f}", ha="center", fontsize=9)
ax.set_xlabel("Household income quintile (Q1 = poorest)"); ax.set_ylabel("CHE 40% (%)")
save_fig(fig, "fig02_che_by_quintile")

# ---- Figure 3: health shock vs no shock ----
ps = p[p["shock_any"].notna()]
sh_rows = []
for sh, lab in [(0, "No shock"), (1, "Health shock")]:
    g = ps[ps.shock_any == sh]
    sh_rows.append([lab, 100 * wmean(g, "che25"), 100 * wmean(g, "che40"), g["oop"].median()])
shock_tab = pd.DataFrame(sh_rows, columns=["group", "CHE25", "CHE40", "median_oop_krw"])
shock_tab.to_csv(f"{TAB}/t02_shock_che.csv", index=False)
fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(2); wd = 0.36
ax.bar(x - wd / 2, shock_tab["CHE25"], wd, color="0.35", edgecolor="0.3", label="CHE 25%")
ax.bar(x + wd / 2, shock_tab["CHE40"], wd, color="0.7", edgecolor="0.3", label="CHE 40%")
for i in range(2):
    ax.text(i - wd / 2, shock_tab["CHE25"][i] + 0.1, f"{shock_tab['CHE25'][i]:.1f}", ha="center", fontsize=9)
    ax.text(i + wd / 2, shock_tab["CHE40"][i] + 0.1, f"{shock_tab['CHE40'][i]:.1f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(shock_tab["group"]); ax.set_ylabel("Households with CHE (%)"); ax.legend(frameon=False)
save_fig(fig, "fig03_shock_che")

# ---- Figure 4: raw PHI buffer within the health-shock group ----
sh = ps[ps.shock_any == 1]
buf_rows = [["No PHI", 100 * wmean(sh[sh.hh_phi_any == 0], "che40")],
            ["Has PHI", 100 * wmean(sh[sh.hh_phi_any == 1], "che40")]]
buf = pd.DataFrame(buf_rows, columns=["group", "CHE40"])
buf.to_csv(f"{TAB}/t02_phi_buffer_raw.csv", index=False)
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(buf["group"], buf["CHE40"], color=["0.25", "0.65"], edgecolor="0.3", width=0.55)
for b, v in zip(bars, buf["CHE40"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.2, f"{v:.1f}%", ha="center", fontweight="bold")
ax.set_ylabel("CHE 40% within health-shock group (%)"); ax.set_xlabel("Baseline private insurance")
save_fig(fig, "fig04_phi_buffer_raw")

print("Descriptives done.")
print(che_year.round(2).to_string(index=False))
print("shock vs no-shock (CHE25/CHE40):")
print(shock_tab.round(2).to_string(index=False))


Descriptives done.
 year  CHE10  CHE25  CHE40
 2019  12.83   4.05   2.20
 2020  12.25   3.11   1.49
 2021  12.05   2.70   1.03
 2022  12.37   2.42   1.07
 2023  15.95   4.05   1.76
 2024  14.61   3.70   1.54
shock vs no-shock (CHE25/CHE40):
       group  CHE25  CHE40  median_oop_krw
    No shock   2.52   1.09        313200.0
Health shock   8.58   3.60       1613630.0
